### JSON Parsing

In [1]:
import os
import json
os.makedirs("data/json_files", exist_ok=True)

In [2]:
json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"}
            ]
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"}
            ]
        }
    ],
    "departments": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 1000000,
            "team_size": 25
        },
        "data_science": {
            "head": "Sarah Williams",
            "budget": 750000,
            "team_size": 15
        }
    }
}

In [3]:
json_data

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
 'departments': {'engineering': {'head': 'Mike Johnson',
   'budget': 1000000,
   'team_size': 25},
  'data_science': {'head': 'Sarah Williams',
   'budget': 750000,
   'team_size': 15}}}

In [5]:
with open("data/json_files/company_data.json", 'w') as f:
    json.dump(json_data, f, indent=4)

In [11]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
]
with open("data/json_files/events.jsonl", 'w') as f:
    for items in jsonl_data:
        f.write(json.dumps(items)+ '\n')

In [16]:
from langchain_community.document_loaders import JSONLoader

json_loader = JSONLoader(file_path="data/json_files/company_data.json", jq_schema= '.employees[]', text_content=False)
json_docs = json_loader.load()

print(json_docs)
print(f"Length of docs: {len(json_docs)}")
for idx, doc in enumerate(json_docs):
    print(f"Doc content {idx+1}: {doc.page_content}")
    print(f"Doc metadata {idx+1}: {doc.metadata}")

[Document(metadata={'source': '/Users/simransingh/Desktop/RAG Projects/0-DataIngestParing/data/json_files/company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': '/Users/simransingh/Desktop/RAG Projects/0-DataIngestParing/data/json_files/company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}')]
Length of docs: 2
Doc content 1: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed

In [ ]:
from typing import List
from langchain_core.documents import Document

def process_json(path)-> List[Document]:
    with open(path, 'r') as f: 
        data = json.load(f)
    documents = []

    for employee in data.get('employees', []):
        content = f"""Employee Profile:
        Name: {employee['name']}
        Role: {employee['role']}
        Skills: {", ".join(employee['skills'])}
        Projects:"""
        for project in employee['projects']:
            content += f"\nName: {project['name']} Status: {project['status']}"
        json_doc = Document(
            page_content= content,
            metadata = {
                'source': path,
                'data_type': 'employee_profile',
                'employee_id': employee['id'],
                'employee_name': employee['name'],
                'role': employee['role']
            }
        )
        documents.append(json_doc)
    return documents    

process_json("data/json_files/company_data.json")        


[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='Employee Profile:\n        Name: John Doe\n        Role: Software Engineer\n        Skills: Python, JavaScript, React\n        Projects:Name: RAG System Status: In ProgressName: Data Pipeline Status: Completed'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}, page_content='Employee Profile:\n        Name: Jane Smith\n        Role: Data Scientist\n        Skills: Python, Machine Learning, SQL\n        Projects:Name: ML Model Status: In ProgressName: Analytics Dashboard Status: Planning')]